# Study 874 — IPO-Price Anchoring ⚓

**Do investors anchor on the IPO *offer price*?**

The offer price is the one round number every newly public stock is introduced by. Two pieces
of folklore follow: (1) an **anchoring pull** — a name stretched far above its offer should get
pulled back down, one below pulled back up (forward return *negatively* related to the
gap-from-offer); and (2) a **below-offer drag** — crossing below the offer, the cohort's
collective cost basis, is a persistent weight. We hard-code a curated table of
**44 famous recent US listings** (offer/reference price + first-trade date, public
record) and test both against **market-adjusted** forward returns (name − SPY),
2014-01-31 → 2026-06-30.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `1eaa178051af`); the
live cells run the fast synthetic control. Curation bias (a small, one-dominant-cohort set) is
named on the Signal axis.*


## 1. The idea in one picture

A stock IPOs at, say, $68. That round number sticks. Anchoring says the price gets *pulled* back toward it — expensive-above gets sold, cheap-below gets bought — and loss-aversion lore adds that once you're *below* the offer (everyone who bought the deal is under water) the name carries a drag. We measure the gap `log(price/offer)` every month and ask whether it predicts next month's **market-adjusted** return.

In [1]:
R = dict(anchor_slope=-0.0023, anchor_t=-0.39, below_bps=-56.84, below_t=-0.56,
         below_leg=13.07, above_leg=69.91)
print('anchoring pull  : FM slope %+.4f  (NW t = %+.2f)  <- right sign, no significance'
      % (R['anchor_slope'], R['anchor_t']))
print('below-offer drag: %+.1f bps/mo  (NW t = %+.2f)'
      % (R['below_bps'], R['below_t']))
print('   below-offer basket %+.1f vs above-offer basket %+.1f bps/mo (market-adj)'
      % (R['below_leg'], R['above_leg']))

anchoring pull  : FM slope -0.0023  (NW t = -0.39)  <- right sign, no significance
below-offer drag: -56.8 bps/mo  (NW t = -0.56)
   below-offer basket +13.1 vs above-offer basket +69.9 bps/mo (market-adj)


## 2. The trap: a loud number with a quiet *t*

Below-offer names trailed above-offer names by **57 bps/mo** market-adjusted — that *looks* like a real drag. But the Newey-West *t* is **-0.56**: with ~45 names that mostly IPO'd in one 2020-21 wave and rose and crashed together, there just isn't enough independent information to call it. The desk's whole job is to separate a big number from a *significant* one.

## 3. Is the pipeline honest? A live synthetic control

We plant an anchoring pull in a seeded toy world (`edge>0`, forward return reverts toward the anchor) and check the detector recovers it — and stays *silent* on the null (`edge=0`, gap present but unpriced). No network.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # the study package
sys.path.insert(0, os.path.abspath("../../.."))    # repo root
import numpy as np
from ipo_anchor import data, strategy as st
null = st.synthetic_control(0.0, n_seeds=20)
planted = st.synthetic_control(0.15, n_seeds=20)
print('null world   : mean FM slope %+.4f  (|t|>=2 in %.0f%% of seeds)'
      % (null['mean_slope'], null['reject_rate']*100))
print('planted world: mean FM slope %+.4f  (|t|>=2 in %.0f%% of seeds)'
      % (planted['mean_slope'], planted['reject_rate']*100))

null world   : mean FM slope +0.0023  (|t|>=2 in 0% of seeds)
planted world: mean FM slope -0.1441  (|t|>=2 in 100% of seeds)


## 4. The honest verdict

The offer price *feels* like an anchor, and the below-offer gap even points the predicted way — but on this curated sample **neither leg clears |t| ≥ 2** (anchoring NW *t* = **-0.39**, drag NW *t* = **-0.56**), the permutation placebo can't tell the slope from noise (two-sided *p* = 0.61), and the tradable book earns a coin-flip gross that dies after borrow and costs. **Signal: None** (underpowered, one-cohort), **Tradability: Mirage**.